# 02 V2 - Immutable causal features and dynamic processed EDA

Run notebook 01 or the prepare CLI first. Set POSEFORGE_RUN explicitly. Features are sealed per clip; all frames, missing masks, original source indices, and capture timestamps are retained. This notebook reads the prepared features and verifies hashes without duplicating them into another feature directory.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path(os.environ['POSEFORGE_ROOT']).expanduser().resolve() if os.environ.get('POSEFORGE_ROOT') else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p/'shared').is_dir() and (p/'scripts').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get('POSEFORGE_DATA', ROOT/'artifacts/dataset_working/Real_Posture_Dataset'))
REPORTS = ROOT/'artifacts/reports'
RUN_VALUE = os.environ.get('POSEFORGE_RUN')
RUN = Path(RUN_VALUE).resolve() if RUN_VALUE else None
SEED = 302
DEVICE = os.environ.get('POSEFORGE_DEVICE', 'cuda:0')
# Optional explicit people; None uses a seeded, quality-eligible subject selection.
VAL_PEOPLE = os.environ.get('POSEFORGE_VAL_SUBJECTS', '').split() or None
TEST_PEOPLE = os.environ.get('POSEFORGE_TEST_SUBJECTS', '').split() or None

def stage(name, *extra):
    command = [sys.executable, str(ROOT/'scripts/train_poseforge.py'), '--stage', name,
               '--data', str(DATA), '--reports', str(REPORTS), '--seed', str(SEED)]
    if name not in ('audit', 'prepare'):
        assert RUN is not None, 'Set POSEFORGE_RUN to a prepared real_vNNN directory'
        command += ['--run', str(RUN), '--device', DEVICE, '--resume']
    subprocess.run(command + list(extra), cwd=ROOT, check=True)

def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
print('Python:',sys.executable,'\nRepository:',ROOT,'\nDataset:',DATA)


In [ ]:
assert RUN is not None, 'Set POSEFORGE_RUN to the prepared real_vNNN directory'
from shared.real_training import verify_seal, TrainingData
from shared.pose_features import FEATURE_VERSION
verify_seal(RUN/'snapshot'); verify_seal(RUN/'processed')
contract=read(RUN/'feature_contract.json')
assert contract['version']==FEATURE_VERSION and contract['flatten_dim']==52
catalog=pd.read_csv(RUN/'catalog.csv')
split_info=read(RUN/'split_manifest.json')
display(pd.Series(contract))

In [ ]:
# Inspect training + validation only. Keep final test features sealed during tuning.
data=TrainingData(RUN,['train','val'],'cpu')
assert data.x.shape[1]==52 and np.isfinite(data.x).all()
x=data.x.reshape(-1,13,4)
assert np.isin(x[:,:,3],(0,1)).all()
assert np.all(x[:,:,:2][x[:,:,3]==0]==0)
assert np.all(x[:,:,2][x[:,:,3]==0]==0)
rows=[]
for c in data.clips:
    rows.append(dict(person=c['person'],label=c['label'],camera=c['camera'],split=c['split'],frames=c['end']-c['start'],eligible=int(data.valid[c['start']:c['end']].sum())))
processed=pd.DataFrame(rows)
display(processed.groupby(['split','label'])[['frames','eligible']].sum())

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(11,4))
processed[processed.split=='train'].groupby('label').eligible.sum().plot.bar(ax=axes[0],title='Training eligible frames')
axes[1].hist(x[:,:,3].sum(1),bins=np.arange(15)-.5);axes[1].set(title='Preserved joint visibility',xlabel='Valid joints',ylabel='Frames')
plt.tight_layout();plt.show()

Preparation validates prefix invariance. tests/test_real_training.py compares the stored causal stream with the deployment pipeline across long absences and track changes. Confidence threshold is 0.35; scale EMA is 0.25. No extra keypoint smoothing or future-frame imputation is used.